In [ ]:
import pickle 
import numpy as np
#with open("runs/QACI_graph_optim__more_complex_128_search_8_envs__1__1716273004/saves/data-768000.pkl","rb") as reader:
with open("runs/QACI_graph_optim__more_complex_NO_range_128_search_8_envs_ent_1e-5__1__1716979604/saves/data-153600.pkl","rb") as reader:
#with open("bench_results/results_test_1718008732_more_complex_large_1024_1/results.pkl","rb") as reader:
    l = pickle.load(reader)
[[x[1].get("applied_rule",None) for x in k[0]] for k in l]

In [ ]:
choice=2
from zx_env.circuit_utils import circuit_extractor



def extract_optimal_path(t):
    # get the optimal index
    idx=np.argmax(t.rewards)
    # get the node associated with it
    transform_steps = [(t.zx_states[idx],t.infos[idx])]
    ste=[idx]
    while idx != 0:
        for idx_tmp, c in enumerate(t.children):
            if idx in c:
                ste.append(idx_tmp)
                idx = idx_tmp
                break
        transform_steps.append((t.zx_states[idx],t.infos[idx]))
    # flip to turn backtracking into "forward tracking"
    print(ste)
    return (transform_steps[::-1],t),ste[::-1]

(p,q),ste=extract_optimal_path(l[choice][1])
print(l[choice][1].rewards)
print([l[choice][1].rewards[s] for s in ste])
print(np.argmax(l[choice][1].rewards))
print(circuit_extractor.extract_circuit(l[choice][1].zx_states[5])[0].stats_dict(depth=True))
l[choice][1].zx_states[0].num_edges()

In [ ]:
import pyzx as zx
from zx_env.circuit_utils import circuit_extractor
from zx_env.general_utils import reward_functions
k= [x[0] for x in p]
ac= [x[1].get("applied_rule",None) for x in p]
res = l[choice][1].rewards
for (y,a,s) in zip(k,ac,ste):
    print("="*100)
    print(a)
    zx.draw(y.copy())
    g=y.copy()
    zx.full_reduce(g)
    g=zx.extract_circuit(g)
    zx.draw(g)
    ex,d=circuit_extractor.extract_circuit(y.copy())
    print("agent achieved",ex.stats_dict()["twoqubit"], "with extraction func", d, ". Full reduce two-qbit would achieve",g.stats_dict()["twoqubit"])
    print("reward received", res[s])
    #print("reward baseline 1", reward_functions.normalized_cnot_count_reward(y,None,1,))
    print("="*100)

In [ ]:
import torch_geometric.utils
import networkx as nx
import matplotlib.pyplot as plt
import random

zx.draw(k[0])
G=torch_geometric.utils.to_networkx(l[0][-1].nodes[0])
limits = plt.axis("off")  # turn off axis
plt.figure(figsize=(6, 7))
G.remove_nodes_from(list(nx.isolates(G)))
colors = {node: "#{:06x}".format(random.randint(0, 0xFFFFFF)) for node in G.nodes()}

# Assign colors to nodes
nx.set_node_attributes(G, colors, 'color')

# Get color values for plotting
node_colors = [colors[node] for node in G.nodes()]
layout=nx.circular_layout(G)
nx.draw_kamada_kawai(G,node_size=16,node_color=node_colors)

In [ ]:
import networkx as nx
from networkx.drawing.nx_pydot import graphviz_layout
import pydot
from matplotlib import pyplot as plt
from matplotlib import cm
import matplotlib as mpl
import numpy as np



G=nx.Graph()
print(max(l[0][1].rewards),np.argmax(l[0][1].rewards))
color=[]
colors = cm.ScalarMappable(mpl.colors.Normalize(vmin=0,vmax=1), plt.get_cmap("cool"))
coloring_function = lambda x: "#%02x%02x%02x" % colors.to_rgba(x/len(G.nodes),bytes=True)[:-1]
for i,js in enumerate(l[choice][1].children):
    if i==0:
        color.append("red")
    else:
        color.append(coloring_function(i))
    for j in js:
        G.add_edge(i,j)
pos = nx.spring_layout(G)
fig,ax, = plt.subplots(figsize=(15, 15))
nx.draw(G,pos,node_color=color,with_labels=True,ax=ax)


In [ ]:
import torch
import torch_geometric
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Batch
from model import BundleNet
import seaborn as sns

path ="runs/QACI_graph_optim__more_complex_NO_range_128_search_8_envs_ent_1e-5__1__1716979604/saves/model-153600.pth"
weights = torch.load(path)
tree = l[choice][1]
agent = BundleNet(
        16,
        5,
        32,
        4,
        4,
        device="cpu",
        model_type = "GAT",
    )
print("before",agent.nodenet.value_projection[0].weight)
agent.load_state_dict(weights)
print("after",agent.nodenet.value_projection[0].weight)
nodes = Batch.from_data_list(tree.nodes).to("cpu")
weights, values = agent.treenet(nodes)
y,x=weights.detach().numpy(), [s.num_vertices()+s.num_edges() for s in tree.zx_states]
sns.lineplot(x=x,y=y)

In [ ]:
for s in ste:
    #print(s,circuit_extractor.extract_circuit(l[0][1].zx_states[s])[0].stats())
    #g=circuit_extractor.extract_circuit(l[0][1].zx_states[s])[0].stats()
    g=l[0][1].zx_states[s].copy()
    zx.full_reduce(g)
    g=zx.extract_circuit(g)
    print(s,g.stats())
    #print(s,l[0][1].zx_states[s].stats())


In [ ]:
import zx_env

env=zx_env.zx_env(
        mutate_probability = 0,
        mutation_steps = 0,
        reward_fn = "absolute_cnot_count_reward",
        n_qubits=5,
        depth = 50,
        mq_ratio=0.6,
        t_ratio=0.2,
        h_ratio=0.2,
        max_steps= 2,
    )
o,_=env.reset()

In [ ]:

zx.draw(o[2])
zx.full_reduce(env.state_zx_graph.clone())
circuit = zx.extract_circuit(env.state_zx_graph.clone())
zx.draw(env.state_zx_graph)


In [ ]:
from zx_env.circuit_utils.circuit_generator import random_circuit
import pyzx as zx
import os
import pickle
n_qubits = 50
n_depth = 2000
mq_ratio= 1.0
h_ratio= 0.0
t_ratio = 0.0
p_x = 1-(mq_ratio+h_ratio+(t_ratio/2))
circuit = random_circuit(n_qubit=n_qubits, num_gates=n_depth, p_two_qubit=mq_ratio, p_H=h_ratio, 
        p_z=t_ratio/2, p_x=p_x, clifford_plus_T=True)

In [ ]:
def CZ_to_CX(circ):
    graph = zx.Circuit(circ.qubits)
    
    for g in circ.gates:
        if isinstance(g, zx.gates.CZ) and g.name=="CZ":
            tar = g.target
            con = g.control
            graph.add_gate("HAD", con)
            graph.add_gate("CNOT", tar,con)
            graph.add_gate("HAD", con)
        else:
            graph.add_gate(g)
    return graph
p = os.listdir("optimized")
for k in p:
    with open(f"optimized/{k}","rb") as reader:
        before=pickle.load(reader)
    #print(before)
    #zx.draw(before)
    #zx.draw(CZ_to_CX(5,before.gates))
    if not before.verify_equality(CZ_to_CX(before)):
        print("not equal after transform")
        zx.draw(before)
        zx.draw(CZ_to_CX(before))

In [ ]:
import bqskit
from bqskit.compiler import Workflow
from bqskit.passes import QuickPartitioner, ForEachBlockPass, ScanningGateRemovalPass, UnfoldPass,ClusteringPartitioner, ScanPartitioner, ExhaustiveGateRemovalPass
import bqskit
from bqskit.ext import qiskit_to_bqskit
from bqskit.compiler import Compiler
from bqskit.compiler.basepass import BasePass
from bqskit.ir.gates import CircuitGate, CNOTGate, CZGate
from zx_compiler import PrintCNOTsPass, ConvertZX, cnot_count_scorer, OutputGraphImage
import logging
zx.draw(circuit)
import pickle 
import numpy as np
import torch
from model import BundleNet
from bqskit.ir.lang.qasm2 import OPENQASM2Language
import os
from bqskit.ext import bqskit_to_qiskit
from bqskit.ir.gates import CZGate
import time


#with open("runs/QACI_graph_optim__more_complex_128_search_8_envs__1__1716273004/saves/data-768000.pkl","rb") as reader:
#with open("runs/QACI_graph_optim__20MIO_32_envs_changed_rules_optimized_5qubit_multirange_4_depth_16__1__1726148732/saves/model-819200.pth","rb") as reader:
#with open("bench_results/results_test_1718008732_more_complex_large_1024_1/results.pkl","rb") as reader:
#    l = pickle.load(reader)
t0=time.time()
with open("tmp.qasm","w+") as writer:
    writer.write(circuit.to_qasm())
bqCirc = bqskit.Circuit.from_file("tmp.qasm")

basic_gate_deletion_workflow = Workflow([
    #ScanPartitioner(5,scoring_fn=cnot_count_scorer),  # Partition into 3-qubit blocks
    #ClusteringPartitioner(5),
    QuickPartitioner(5),
    #ForEachBlockPass(PrintCNOTsPass()),
    ForEachBlockPass(ScanningGateRemovalPass()),  # Apply gate deletion to each block (in parallel)
    ForEachBlockPass(ConvertZX()),
    ForEachBlockPass(ScanningGateRemovalPass()),  # Apply gate deletion to each block (in parallel)
    UnfoldPass(),  # Unfold the blocks back into the original circuit 
    #####
    #ScanPartitioner(5,scoring_fn=cnot_count_scorer),  # Partition into 3-qubit blocks
    #ClusteringPartitioner(5),
    #QuickPartitioner(5),
    #ForEachBlockPass(OutputGraphImage()),
    #ForEachBlockPass(ScanningGateRemovalPass()),  # Apply gate deletion to each block (in parallel)
    #ForEachBlockPass(PrintCNOTsPass()),
    #ForEachBlockPass(ConvertZX()),
    #UnfoldPass(),  # Unfold the blocks back into the original circuit
])
opt_circuit = bqCirc
circs=[bqCirc.copy()]
cnotcount=[]
for _ in range(10):
    with Compiler() as compiler:
        opt_circuit = compiler.compile(opt_circuit, workflow=basic_gate_deletion_workflow)
        opt_circuit.save("compiled.qasm")
        opt_circuit = CZ_to_CX(zx.Circuit.from_qasm_file("compiled.qasm"))
        qasm = opt_circuit.to_qasm()
        opt_circuit = OPENQASM2Language().decode(qasm)
        circs.append(opt_circuit.copy())
        cnotcount.append(opt_circuit.count(CNOTGate())+opt_circuit.count(CZGate()))
print("POST OPT",cnotcount)
opt_circuit.save("compiled.qasm")  
zx.draw(zx.Circuit.from_qasm_file("compiled.qasm"))
print("time taken", time.time()-t0)
from qiskit.compiler import transpile
just_transpile=transpile(bqskit_to_qiskit(bqCirc),optimization_level=3)
ours_and_transpile=transpile(bqskit_to_qiskit(opt_circuit),optimization_level=3)
if not os.path.exists("data_processing/"):
    os.mkdir("data_processing/")
with open(f"data_processing/{int(time.time())}.pkl", "wb+") as writer:
    pickle.dump({"just_transpile":just_transpile, "ours_and_transpile":ours_and_transpile, "our_optimization": circs}, writer)


In [ ]:
from bqskit.ext import bqskit_to_qiskit
from bqskit.ir.gates import CZGate

print(bqCirc.count(CNOTGate()))
print(opt_circuit.count(CNOTGate()),opt_circuit.count(CZGate()))

In [ ]:
from zx_env.general_utils.utils import check_equality

zx.draw(zx.extract_circuit(circuit.copy().to_graph()).to_graph())
zx.draw(circuit.copy().to_graph())
zx.extract_circuit(circuit.copy().to_graph()).verify_equality(circuit)

In [ ]:
bqskit_to_qiskit(opt_circuit,).draw(output="mpl")

In [ ]:
from qiskit.compiler import transpile
m=transpile(bqskit_to_qiskit(bqCirc),optimization_level=3)
m.count_ops() 

In [ ]:
from qiskit.compiler import transpile
k=transpile(bqskit_to_qiskit(opt_circuit),optimization_level=3)
k.count_ops() 

In [ ]:
import pickle
import os
import pyzx as zx
from zx_env.general_utils.utils import check_equality
from zx_env import extract_circuit


p = os.listdir("original")
def read_and_compare(name):
    with open(f"original/{name}","rb") as reader:
            before=pickle.load(reader)
    with open(f"optimized/{name}","rb") as reader:
        after=pickle.load(reader)
    print("before",before)
    zx.draw(before)
    print("\n"*5)
    print("After")
    #zx.draw(after)
    print(after.verify_equality(before))
#print(p)
for i in p:
    read_and_compare(i)

In [ ]:
total=[]
for name in os.listdir("original"):
    with open(f"original/{name}","rb") as reader:
        before=pickle.load(reader)
    o = zx.optimize.Optimizer(before)
    before,corrs = o.parse_circuit(do_swaps=True,separate_correction=True)
    total+=corrs
len(total)

In [ ]:
for name in os.listdir("optimized"):
    with open(f"optimized/{name}","rb") as reader:
        before=pickle.load(reader)
    print(before.gates)


In [ ]:
c = zx.Circuit(2)
c.add_gate("CNOT", control=0, target=1)
c.add_gate("CNOT", control=1, target=0)
c.add_gate("CNOT", control=0, target=1)
o = zx.optimize.Optimizer(c)
d, corr = o.parse_circuit(do_swaps=True,separate_correction=True)
zx.draw(c)
zx.draw(d)

In [ ]:
os.listdir("data_processing")
with open(f"data_processing/{os.listdir('data_processing')[-1]}", "rb") as reader:
    k=pickle.load(reader)
k["ours_and_transpile"].draw()